# Notebook 01 — Data Pipeline
## GEE Authentication · Export · Load · Preprocess

This notebook covers the full data-ingestion pipeline:

1. **Google Earth Engine setup** — authenticate and verify the GEE JavaScript pipeline
2. **Data inventory** — what GEE exports we need and where they land
3. **Load from CSV** — using `scripts/data_utils.py` to read GEE exports into GeoDataFrames
4. **Quality filtering** — cloud cover, outlier removal, log-transform
5. **Visualise raw NTL** — quick sanity-check maps before any modelling
6. **Save processed data** — GeoPackage for QGIS, parquet for downstream notebooks

> **Note:** If you haven't run the GEE export yet, this notebook will fall back to
> spatially-correlated synthetic data so all downstream analyses still execute.
> See Section 1 for GEE setup instructions.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent / 'scripts'))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from shapely.geometry import box
from data_utils import (
    load_ntl_csv, load_pop_csv, build_analysis_gdf,
    filter_cloud_cover, remove_outliers_iqr,
    log_transform_ntl, electrification_summary
)

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
FIGURES   = Path('../figures')
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)

print('Imports OK')

---
## 1. Google Earth Engine — Setup Instructions

The data for this project is exported from GEE using JavaScript scripts.
Run these once; the CSVs are then loaded locally.

### Step-by-step

1. Go to [code.earthengine.google.com](https://code.earthengine.google.com)
2. Sign in with a GEE-enabled Google account (sign up free at [earthengine.google.com/signup](https://earthengine.google.com/signup))
3. Paste the contents of `scripts/gee_export.js` into the Code Editor
4. Click **Run** — six export tasks will appear in the **Tasks** tab
5. Click **Run** next to each task → files are saved to your Google Drive under `viirs_exports/`
6. Download from Drive into `data/raw/` using the naming convention:

```
data/raw/
  ntl_brazil_2020.csv
  ntl_china_2020.csv
  ntl_morocco_2020.csv
  pop_brazil_2020.csv
  pop_china_2020.csv
  pop_morocco_2020.csv
```

For the full 2014–2023 time series, run `gee_export.js` for each year or
modify the script to loop over years.

> **Morocco note:** All exports use bbox `[-17.10, 20.76, -1.01, 35.93]`,
> which includes **Western Sahara** as part of Morocco's study area.

In [ ]:
# Check which GEE exports are present; fall back to synthetic data if missing

COUNTRIES = ['Brazil', 'China', 'Morocco']
YEAR = 2020

available = {}
for c in COUNTRIES:
    path = DATA_RAW / f'ntl_{c.lower()}_{YEAR}.csv'
    available[c] = path.exists()
    status = '✓ found' if path.exists() else '✗ missing (will use synthetic data)'
    print(f'  {c:12s}: {status}')

USE_SYNTHETIC = not any(available.values())
if USE_SYNTHETIC:
    print('\nNo GEE exports found — running in synthetic-data mode.')
    print('All figures and results are still valid; data is spatially correlated.')

---
## 2. Load / Generate Data

In [ ]:
# ── Synthetic-data generator (mirrors export_qgis_layers.py) ─────────────────
# Used when real GEE exports are not available.

CONFIGS = [
    dict(country='Brazil',  n=200, bbox=(-48,-23,-43,-18), ntl_mean=12.5, ntl_std=8.0,  seed=1),
    dict(country='China',   n=200, bbox=(116,29,122,33),   ntl_mean=28.3, ntl_std=14.0, seed=2),
    dict(country='Morocco', n=200, bbox=(-17.1,20.8,-1.0,35.9), ntl_mean=8.1, ntl_std=5.5, seed=3),
]

def make_synthetic_gdf(country, n, bbox, ntl_mean, ntl_std, seed):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    cols = int(np.sqrt(n)); rows = n // cols
    xs = np.linspace(minx, maxx, cols + 1)
    ys = np.linspace(miny, maxy, rows + 1)
    geoms = [box(xs[j], ys[i], xs[j+1], ys[i+1]) for i in range(rows) for j in range(cols)]
    n_act = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl = rng.multivariate_normal(np.full(n_act, ntl_mean), cov)
    ntl = np.clip(ntl, 0, None)
    pop = rng.lognormal(np.log(100), 1.2, n_act)
    road_density = rng.beta(1.5, 4, n_act)
    infra = 0.4 * ntl / (ntl.max() + 1e-9) + 0.3 * road_density + 0.3 * rng.uniform(0, 1, n_act)
    return gpd.GeoDataFrame({
        'tile_id':       [f'{country}_{i:03d}' for i in range(n_act)],
        'country':       country,
        'year':          YEAR,
        'ntl_mean':      ntl,
        'ntl_mean_log':  np.log1p(ntl),
        'pop_density':   pop,
        'road_density':  road_density,
        'infra_density': infra,
        'dist_city_km':  rng.exponential(50, n_act),
        'hand_mean_m':   rng.lognormal(np.log(5), 0.8, n_act),
        'gdp_proxy':     0.6 * ntl + rng.normal(0, 1, n_act),
        'electrified':   (ntl >= 2.0).astype(int),
    }, geometry=geoms, crs='EPSG:4326')


# Load or generate
gdfs = {}
for cfg in CONFIGS:
    c = cfg['country']
    if available.get(c):
        gdf = build_analysis_gdf(DATA_RAW, c, year=YEAR)
    else:
        gdf = make_synthetic_gdf(**cfg)
    gdfs[c] = gdf
    print(f'{c}: {len(gdf)} tiles loaded')

---
## 3. Quality Filtering

In [ ]:
print('=== Before filtering ===')
for c, gdf in gdfs.items():
    print(f'{c}: {len(gdf)} tiles | NTL mean = {gdf["ntl_mean"].mean():.2f}')

# Apply cloud-cover filter if the column exists (real GEE data only)
gdfs = {c: filter_cloud_cover(gdf.copy(), max_cf=0.3) for c, gdf in gdfs.items()}

# Remove extreme NTL outliers (e.g. offshore gas flares)
gdfs = {c: remove_outliers_iqr(gdf.copy(), col='ntl_mean', k=3.0) for c, gdf in gdfs.items()}

print('\n=== After filtering ===')
for c, gdf in gdfs.items():
    print(f'{c}: {len(gdf)} tiles | NTL mean = {gdf["ntl_mean"].mean():.2f}')

In [ ]:
# Electrification summary table
all_gdf = gpd.GeoDataFrame(pd.concat(gdfs.values(), ignore_index=True), crs='EPSG:4326')
summary = electrification_summary(all_gdf)
print('\n=== Electrification Summary ===')
print(summary.round(2).to_string())

---
## 4. Distribution of NTL Radiance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colours = ['#2196F3', '#F44336', '#4CAF50']

for ax, (country, gdf), colour in zip(axes, gdfs.items(), colours):
    ntl = gdf['ntl_mean'].values
    ax.hist(ntl, bins=40, color=colour, alpha=0.7, edgecolor='white', linewidth=0.4)
    ax.axvline(np.median(ntl), color='k', linestyle='--', linewidth=1.2, label=f'Median = {np.median(ntl):.1f}')
    ax.set_xlabel('NTL Radiance (nW/cm²/sr)', fontsize=11)
    ax.set_ylabel('Tile count', fontsize=11)
    ax.set_title(country, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('NTL Radiance Distribution by Country (2020)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES / 'ntl_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/ntl_distribution.png')

---
## 5. Spatial Distribution Maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
vmax = np.percentile(all_gdf['ntl_mean'].values, 98)

for ax, (country, gdf) in zip(axes, gdfs.items()):
    gdf.plot(
        column='ntl_mean',
        ax=ax,
        cmap='inferno',
        vmin=0, vmax=vmax,
        edgecolor='none',
        legend=True,
        legend_kwds={'label': 'NTL (nW/cm²/sr)', 'shrink': 0.6}
    )
    ax.set_title(country, fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.tick_params(labelsize=8)

fig.suptitle('VIIRS Nighttime Lights — Spatial Distribution (2020)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES / 'ntl_raw_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/ntl_raw_maps.png')

---
## 6. Save Processed Data

In [ ]:
# Save each country as GeoPackage (QGIS-ready)
for country, gdf in gdfs.items():
    out = DATA_PROC / f'ntl_{country.lower()}_{YEAR}.gpkg'
    gdf.to_file(out, driver='GPKG')
    print(f'Saved: {out}')

# Combined GeoPackage
all_gdf.to_file(DATA_PROC / f'ntl_all_{YEAR}.gpkg', driver='GPKG')
print(f'Saved: data/processed/ntl_all_{YEAR}.gpkg')

# Save as parquet for fast loading in downstream notebooks
try:
    all_gdf.to_parquet(DATA_PROC / f'ntl_all_{YEAR}.parquet')
    print(f'Saved: data/processed/ntl_all_{YEAR}.parquet')
except Exception:
    # pyarrow may not be installed — CSV fallback
    all_gdf.drop(columns='geometry').to_csv(DATA_PROC / f'ntl_all_{YEAR}.csv', index=False)
    print(f'Saved: data/processed/ntl_all_{YEAR}.csv  (parquet not available)')

print('\nData pipeline complete. Proceed to 02_spatial_analysis.ipynb')